### Baseline

In [1]:
# Configuración de recursos según la máquina
entorno = 'casa'  # cambiar a 'oficina' según corresponda

if entorno == 'casa':
    # i7-12700K, 12 cores, 32GB RAM
    duckdb_threads = 8
    duckdb_memory_limit = '16GB'
    rf_n_jobs = 8
    lgb_threads = 8
elif entorno == 'oficina':
    # 16 cores, 63.8GB RAM
    duckdb_threads = 12
    duckdb_memory_limit = '45GB'
    rf_n_jobs = 10
    lgb_threads = 10          # 8 por precaución: la oficina se apagó con CPU al 100% el 27/09
else:
    raise ValueError("entorno debe ser 'casa' u 'oficina'")

print(f"Entorno: {entorno} | DuckDB: {duckdb_threads} threads, {duckdb_memory_limit} | RF: n_jobs={rf_n_jobs} | LGB: {lgb_threads} threads")

Entorno: casa | DuckDB: 8 threads, 16GB | RF: n_jobs=8 | LGB: 8 threads


#### Librerías y Path Variables

In [2]:
# Importando Librerías

import os, sys
os.environ['PATH'] = os.path.join(sys.prefix, 'Library', 'bin') + os.pathsep + os.environ['PATH']
import duckdb

import pandas as pd
import numpy as np
import pickle                                                                   # CAMBIO

import plotly.graph_objects as go
import plotly.express as px

from sklearn.metrics import roc_auc_score                                       # CAMBIO

import lightgbm as lgb                                                          # CAMBIO

import optuna                                                                   # CAMBIO
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_slice, plot_contour   # CAMBIO

In [3]:
# Paths Variables

from pathlib import Path
nb_file = globals().get('__vsc_ipynb_file__')                 # ruta de la notebook (la define VS Code)
cwd = Path(nb_file).parent if nb_file else Path.cwd()         # carpeta de la notebook; si no es VS Code, usa el cwd
RAIZ = (cwd / '../../../../05_ExpMeses').resolve()

sys.path.insert(0, str(cwd))                                  # semillas.py está al lado de la notebook
from semillas import SEMILLAS as semillas

base_path = RAIZ.as_posix() + '/'                             # as_posix: barras normales también en Windows
print(RAIZ, RAIZ.exists(), (cwd / 'semillas.py').exists())

dataset_path = base_path + 'data/'
db_path = base_path + 'databases/'
modelos_path = base_path + 'modelos/'
out_path = base_path + 'out/'
dataset_file = 'competencia_02_meses.parquet'
ruta_meses = dataset_path + dataset_file
print(ruta_meses, Path(ruta_meses).exists())

os.makedirs(db_path, exist_ok=True)                            # CAMBIO: carpeta de las bases de Optuna
os.makedirs(modelos_path, exist_ok=True)                       # CAMBIO: carpeta de modelos

E:\Training\05_Exactas-UBA\06_DM_EconomiaFinanzas\05_ExpMeses True True
E:/Training/05_Exactas-UBA/06_DM_EconomiaFinanzas/05_ExpMeses/data/competencia_02_meses.parquet True


#### Carga de los Meses del Baseline
Se leen de `competencia_02_meses.parquet` los meses 201901 a 201910 y se dejan en una tabla en memoria de DuckDB (`d2019`). 201911 y 201912 son del dashboard y no se usan. Los meses 1 y 2 no se entrenan: solo hacen falta para calcular los lags.

In [4]:
%%time
# Lee del parquet de meses; solo hasta 201910 (201911 y 201912 son del dashboard)
con = duckdb.connect()
con.execute(f"SET threads = {duckdb_threads}")
con.execute(f"SET memory_limit = '{duckdb_memory_limit}'")
con.execute("SET enable_progress_bar = false")

MES_DESDE, MES_HASTA = 201901, 201910

con.execute(f"""
    CREATE OR REPLACE TABLE d2019 AS
    SELECT * FROM read_parquet('{ruta_meses}')
    WHERE foto_mes BETWEEN {MES_DESDE} AND {MES_HASTA}
""")
print(con.execute("SELECT count(*) FROM d2019").fetchall()[0][0], 'filas')

1286487 filas
CPU times: total: 6.17 s
Wall time: 1.12 s


#### Filas, Clientes, Variables y Clase Ternaria por Mes

In [5]:
mes_resumen = con.sql("""
    SELECT foto_mes,
           count(*)                                      AS filas,
           count(DISTINCT numero_de_cliente)             AS clientes,
           count(*) - count(DISTINCT numero_de_cliente)  AS duplicados      -- tiene que dar 0
    FROM d2019
    GROUP BY foto_mes
    ORDER BY foto_mes
""").df()
mes_resumen

,foto_mes,filas,clientes,duplicados
0,201901,123916,123916,0
1,201902,125020,125020,0
2,201903,125622,125622,0
3,201904,126116,126116,0
4,201905,126763,126763,0
5,201906,128370,128370,0
6,201907,129853,129853,0
7,201908,131775,131775,0
8,201909,133419,133419,0
9,201910,135633,135633,0


In [6]:
import math

cols = [r[0] for r in con.execute("SELECT column_name FROM (DESCRIBE d2019)").fetchall()]
print(len(cols), 'columnas')
print('clase_ternaria' in cols)

n_col = 5                                          # cantidad de columnas a mostrar
n_fil = math.ceil(len(cols) / n_col)
relleno = cols + [''] * (n_fil * n_col - len(cols))     # completa con vacíos para que cierre el rectángulo
tabla = pd.DataFrame(np.array(relleno).reshape(n_col, n_fil).T)
tabla.style.hide(axis='index').hide(axis='columns')

155 columnas
True


numero_de_cliente,mtarjeta_master_consumo,mpagodeservicios,mcheques_emitidos_rechazados,Master_mpagado
foto_mes,cprestamos_personales,cpagomiscuentas,tcallcenter,Master_mpagospesos
active_quarter,mprestamos_personales,mpagomiscuentas,ccallcenter_transacciones,Master_mpagosdolares
cliente_vip,cprestamos_prendarios,ccajeros_propios_descuentos,thomebanking,Master_fechaalta
internet,mprestamos_prendarios,mcajeros_propios_descuentos,chomebanking_transacciones,Master_mconsumototal
cliente_edad,cprestamos_hipotecarios,ctarjeta_visa_descuentos,ccajas_transacciones,Master_cconsumos
cliente_antiguedad,mprestamos_hipotecarios,mtarjeta_visa_descuentos,ccajas_consultas,Master_cadelantosefectivo
mrentabilidad,cplazo_fijo,ctarjeta_master_descuentos,ccajas_depositos,Master_mpagominimo
mrentabilidad_annual,mplazo_fijo_dolares,mtarjeta_master_descuentos,ccajas_extracciones,Visa_delinquency
mcomisiones,mplazo_fijo_pesos,ccomisiones_mantenimiento,ccajas_otras,Visa_status
mactivos_margen,cinversion1,mcomisiones_mantenimiento,catm_trx,Visa_mfinanciacion_limite


In [7]:
# Clase por mes. Esperado: 201911 solo con BAJA+1, 201912 sin clase
clase_mes = (con.sql("""
    SELECT foto_mes, coalesce(clase_ternaria, '(sin clase)') AS clase, count(*) AS filas
    FROM d2019
    GROUP BY 1, 2
    ORDER BY 1, 2
""").df().pivot(index='foto_mes', columns='clase', values='filas').fillna(0).astype(int))

total = clase_mes.sum(axis=1)
clase_mes['pct_BAJA+2'] = (clase_mes.get('BAJA+2', 0) / total * 100).round(3)
clase_mes

clase,BAJA+1,BAJA+2,CONTINUA,pct_BAJA+2
foto_mes,,,,
201901,647,697,122572,0.562
201902,700,696,123624,0.557
201903,700,774,124148,0.616
201904,776,575,124765,0.456
201905,576,658,125529,0.519
201906,665,589,127116,0.459
201907,595,656,128602,0.505
201908,659,543,130573,0.412
201909,548,615,132256,0.461


#### Tipos de Variables

In [8]:
esquema = con.sql("DESCRIBE d2019").df()[['column_name', 'column_type']]
ids = ['numero_de_cliente', 'foto_mes']

tipos_num = ('TINYINT', 'SMALLINT', 'INTEGER', 'BIGINT', 'HUGEINT', 'UTINYINT', 'USMALLINT', 'UINTEGER', 'UBIGINT',
             'FLOAT', 'DOUBLE', 'DECIMAL')
es_num = esquema['column_type'].str.startswith(tipos_num)

cols_num = [c for c in esquema.loc[es_num, 'column_name'] if c not in ids]            # a las que se les mide nulos y ceros
cols_otras = [c for c in esquema.loc[~es_num, 'column_name'] if c not in ids]         # texto, fechas, etc.

print(len(esquema), 'columnas |', len(cols_num), 'numéricas |', len(cols_otras), 'no numéricas')
print('No numéricas:', cols_otras)           # clase_ternaria y, si aparecen, columnas 0/1 que llegaron como texto
esquema['column_type'].value_counts()

155 columnas | 152 numéricas | 1 no numéricas
No numéricas: ['clase_ternaria']


column_type
BIGINT     80
DOUBLE     74
VARCHAR     1
Name: count, dtype: int64

#### Feature Engineering -- Lags y Deltas 1 y 2

In [9]:
%%time
# Feature engineering: lag 1 y 2 y delta 1 y 2 de las variables numéricas, calculados por mes real
ruta_fe = dataset_path + 'competencia_02_fe.parquet'        # parquet de salida (no es el de entrada, para no pisarlo)

# Índice de mes continuo (año*12 + mes): permite restar 1 o 2 y cruzar de diciembre a enero sin saltos
mes_idx = "(foto_mes // 100 * 12 + foto_mes % 100)"
cols_sql = ', '.join(f'"{c}"' for c in cols_num)            # las 152 numéricas, entre comillas, para el SELECT

# Una columna nueva por variable y tipo: lag_1, lag_2 (valor de hace 1 y 2 meses) y delta_1, delta_2 (valor actual menos el lag)
sel = []
for c in cols_num:
    sel.append(f'p1."{c}" AS "lag_1_{c}"')
    sel.append(f'p2."{c}" AS "lag_2_{c}"')
    sel.append(f'd."{c}" - p1."{c}" AS "delta_1_{c}"')
    sel.append(f'd."{c}" - p2."{c}" AS "delta_2_{c}"')

query = f"""
    -- base: la tabla de 2019 con el índice de mes; prev: el mismo cliente y mes, solo con las variables a "mirar para atrás"
    WITH base AS (SELECT *, {mes_idx} AS mes_idx FROM d2019),
         prev AS (SELECT numero_de_cliente, mes_idx, {cols_sql} FROM base)
    -- d.*: todas las columnas originales (sin el índice auxiliar) más los lags y deltas
    SELECT d.* EXCLUDE (mes_idx), {', '.join(sel)}
    FROM base d
    -- mismo cliente, un mes antes (p1) y dos meses antes (p2); LEFT JOIN: si no existe ese mes, el lag queda nulo
    LEFT JOIN prev p1 ON p1.numero_de_cliente = d.numero_de_cliente AND p1.mes_idx = d.mes_idx - 1
    LEFT JOIN prev p2 ON p2.numero_de_cliente = d.numero_de_cliente AND p2.mes_idx = d.mes_idx - 2
"""
# Se escribe directo a parquet (comprimido con ZSTD), con orden fijo de filas
con.execute(f"""
    COPY ({query} ORDER BY d.foto_mes, d.numero_de_cliente)
    TO '{ruta_fe}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
print('guardado:', ruta_fe, round(Path(ruta_fe).stat().st_size / 1e6), 'MB')

guardado: E:/Training/05_Exactas-UBA/06_DM_EconomiaFinanzas/05_ExpMeses/data/competencia_02_fe.parquet 785 MB
CPU times: total: 3min 25s
Wall time: 1min 16s


In [10]:
# 1) filas, clientes y columnas (esperadas: las de d2019 + 4 por cada variable numérica)
n_fil, n_cli = con.execute(f"SELECT count(*), count(DISTINCT numero_de_cliente) FROM read_parquet('{ruta_fe}')").fetchall()[0]
n_col = len(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{ruta_fe}')").fetchall())
print(n_fil, 'filas |', n_cli, 'clientes |', n_col, 'columnas | esperadas:', len(esquema) + 4 * len(cols_num))

# 2) % de nulos de los lags de una variable, por mes
con.execute(f"""
    SELECT foto_mes,
           round(100 * avg((lag_1_mcuentas_saldo IS NULL)::int), 2) AS pct_null_lag1,
           round(100 * avg((lag_2_mcuentas_saldo IS NULL)::int), 2) AS pct_null_lag2
    FROM read_parquet('{ruta_fe}') GROUP BY 1 ORDER BY 1
""").df()

1286487 filas | 141458 clientes | 763 columnas | esperadas: 763


,foto_mes,pct_null_lag1,pct_null_lag2
0,201901,100.00,100.00
1,201902,1.40,100.00
2,201903,1.04,2.43
3,201904,0.95,1.98
4,201905,1.12,2.06
5,201906,1.70,2.81
6,201907,1.65,3.33
7,201908,1.91,3.53
8,201909,1.73,3.61
9,201910,2.04,3.72


##### Cargado de Datos para LightGBM

In [11]:
%%time
# Carga de los meses del baseline desde el parquet de FE
meses_opt_train  = [201903, 201904, 201905, 201906]     # optimización: train
meses_opt_val    =  201908                              # optimización: validación
meses_test_train = [201905, 201906, 201907, 201908]     # reentrenamiento para el test
meses_test       =  201910                              # test

MESES = sorted(set(meses_opt_train + [meses_opt_val] + meses_test_train + [meses_test]))
lista_meses = ', '.join(str(m) for m in MESES)

data = con.execute(f"""
    SELECT * FROM read_parquet('{ruta_fe}')
    WHERE foto_mes IN ({lista_meses})
    ORDER BY foto_mes, numero_de_cliente
""").fetchdf()                                           # orden fijo de filas

print(data.shape)

# Filas por mes
print('\nFilas por mes:')
print(data['foto_mes'].value_counts().sort_index().to_string())

# Resumen de los bloques de meses, con sus filas
def filas(meses):
    meses = meses if isinstance(meses, list) else [meses]
    return int(data['foto_mes'].isin(meses).sum())

bloques = {
    'Optimización | train':      meses_opt_train,
    'Optimización | validación': meses_opt_val,
    'Test         | train':      meses_test_train,
    'Test         | test':       meses_test,
}
print('\nBloques de meses:')
for nombre, meses in bloques.items():
    print(f'{nombre:27s}: {meses}  ->  {filas(meses):,} filas')

(904132, 763)

Filas por mes:
foto_mes
201903    125622
201904    126116
201905    126763
201906    128370
201907    129853
201908    131775
201910    135633

Bloques de meses:
Optimización | train       : [201903, 201904, 201905, 201906]  ->  506,871 filas
Optimización | validación  : 201908  ->  131,775 filas
Test         | train       : [201905, 201906, 201907, 201908]  ->  516,761 filas
Test         | test        : 201910  ->  135,633 filas
CPU times: total: 55.9 s
Wall time: 30.5 s


##### Clase Binaria (BAJA+1 y BAJA+2 contra CONTINUA) + X/y de Optimización

In [12]:
# Clase binaria: BAJA+1 y BAJA+2 juntas (1) contra CONTINUA (0)
print('Filas sin clase:', int(data['clase_ternaria'].isna().sum()))      # tiene que dar 0 en estos meses
data['clase_binaria_b12'] = np.where(data['clase_ternaria'].isin(['BAJA+1', 'BAJA+2']), 1, 0)

# No entran a X: la clase, la binaria, el id y el mes
cols_descartar = ['clase_ternaria', 'clase_binaria_b12', 'numero_de_cliente', 'foto_mes']

# Optimización: train (meses 3 a 6) y validación (mes 8)
train_data = data[data['foto_mes'].isin(meses_opt_train)]
val_data   = data[data['foto_mes'] == meses_opt_val]

X_train = train_data.drop(columns=cols_descartar)
y_train = train_data['clase_binaria_b12']

X_val = val_data.drop(columns=cols_descartar)
y_val = val_data['clase_binaria_b12']

# feature_pre_filter=False: necesario porque Optuna va a variar min_data_in_leaf entre trials
train_data_lgb = lgb.Dataset(X_train, label=y_train, params={'feature_pre_filter': False})
val_data_lgb   = lgb.Dataset(X_val, label=y_val, reference=train_data_lgb)

print('train:', X_train.shape, '| positivos', int(y_train.sum()), f'({100 * y_train.mean():.2f}%)')
print('valid:', X_val.shape,   '| positivos', int(y_val.sum()),   f'({100 * y_val.mean():.2f}%)')

Filas sin clase: 0
train: (506871, 760) | positivos 5313 (1.05%)
valid: (131775, 760) | positivos 1202 (0.91%)


#### Optimización de Hiperparámetros (Optuna, Validación en 201908)

In [13]:
# Optimización bayesiana (Optuna): rangos del baseline.yaml, validación en el mes 201908
def objective(trial):

    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'boost_from_average': True,
        'feature_pre_filter': False,
      # bayesianos
        'num_leaves': trial.suggest_int('num_leaves', 8, 256),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 20, 5000, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.1, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.3, 1.0),
      # fijos
        'bagging_freq': 1,
        'seed': semillas[0],
        'num_threads': lgb_threads,
        'verbose': -1
    }

    model = lgb.train(
        params,
        train_data_lgb,
        num_boost_round=5000,                                           # techo; el corte lo pone el early stopping
        valid_sets=[val_data_lgb],                                      # validación en 201908 
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )

    trial.set_user_attr('best_iter', model.best_iteration)              # iteraciones del mejor AUC
    return model.best_score['valid_0']['auc']                           # AUC en validación: ordena los trials

storage_name = "sqlite:///" + db_path + "ExpMeses_Optimization.db"
study_name = "EM_Baseline_b12_T201903t06V201908"

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=semillas[0]),               # sampler con semilla, reproducible
    study_name=study_name,
    storage=storage_name,
    load_if_exists=True,
)

print(len(study.trials))     # debe dar 0 si la base es nueva

[I 2026-11-15 17:29:25,542] Using an existing study with `study_name='EM_Baseline_b12_T201903t06V201908'` instead of creating a new one.


50


In [14]:
objetivo_total = 50                                        # número de trials que queremos correr en total
faltan = objetivo_total - len(study.trials)
print(f"Tengo {len(study.trials)}, corro {faltan} más")
if faltan > 0:
    study.optimize(objective, n_trials=faltan)

Tengo 50, corro 0 más


##### Resultados Optimización de Hiperparámetros

In [15]:
# Historia de la optimización: AUC de cada trial y mejor AUC acumulado
optuna.visualization.plot_optimization_history(study)

In [16]:
# Importancia de cada hiperparámetro: cuánto explica de las diferencias de AUC entre trials
plot_param_importances(study)

In [17]:
# Slice: AUC de cada trial según el valor de cada hiperparámetro (uno por gráfico)
plot_slice(study)

In [18]:
# Contour de un solo par: learning_rate contra min_data_in_leaf (el freno del baseline)
plot_contour(study, params=['learning_rate', 'min_data_in_leaf'])

##### Mejor Trial

In [19]:
mejor = study.best_trial
print('Mejor trial:', mejor.number, '| AUC validación:', round(mejor.value, 4), '| best_iter:', mejor.user_attrs['best_iter'])
mejor.params

Mejor trial: 32 | AUC validación: 0.9325 | best_iter: 440


{'num_leaves': 119,
 'learning_rate': 0.01398408563656187,
 'min_data_in_leaf': 920,
 'feature_fraction': 0.3380419059507952,
 'bagging_fraction': 0.5048862447947764}

#### Train 201905-201908 y Test 201910 con 5 Semillas

##### Variables para Reentrenar y Testear

In [20]:
# Test: train con los meses 5 a 8 y test en el mes 10
data_test_train = data[data['foto_mes'].isin(meses_test_train)]       # filas de 201905 a 201908 (incluye agosto, el mes de validación)
data_test       = data[data['foto_mes'] == meses_test]                # filas de 201910 (para testear el modelo final)

# X: variables sin la clase, el id y el mes (cols_descartar)
X_test_train = data_test_train.drop(columns=cols_descartar)
y_test_train = data_test_train['clase_binaria_b12']                   # train: target fusionado (BAJA+1 y BAJA+2 = 1, CONTINUA = 0)
X_test       = data_test.drop(columns=cols_descartar)
y_test = data_test['clase_ternaria']                            # clase ternaria de octubre, solo para medir con el BAJA+2 real; no entra en X

# Dataset de LightGBM con el train del test; feature_pre_filter=False por consistencia con la optimización
data_test_train_lgb = lgb.Dataset(X_test_train, label=y_test_train, params={'feature_pre_filter': False})

# Chequeo: forma de cada matriz y cantidad de positivos
print('train:', X_test_train.shape, '| positivos (fusionado)', int(y_test_train.sum()), f'({100 * y_test_train.mean():.2f}%)')                                          # CAMBIO: texto
print('test :', X_test.shape,       '| BAJA+2', int((y_test == 'BAJA+2').sum()), f'({100 * (y_test == "BAJA+2").mean():.2f}%)')                          # CAMBIO: cuenta BAJA+2 en test

train: (516761, 760) | positivos (fusionado) 4941 (0.96%)
test : (135633, 760) | BAJA+2 632 (0.47%)


##### Reentrenamiento y Testeo con AUC y Ganancia

In [21]:
# Parámetros de la ganancia
ganancia_acierto = 1_072_500                              # ganancia por cliente BAJA+2 enviado
costo_estimulo   = 27_500                                 # costo por cliente enviado que no es BAJA+2

# Ganancia con el BAJA+2 real: se acierta solo si el cliente fue BAJA+2, aunque el modelo haya aprendido las dos bajas
def ganancia_real(y_pred, y_true_class, umbral=0.025):
    ganancia = np.where(y_true_class == 'BAJA+2', ganancia_acierto, -costo_estimulo)
    return ganancia[y_pred >= umbral].sum()

In [22]:
n_semillas_test = 5                                       # cantidad de semillas: sirve para medir cuánto varía la ganancia por azar
best_iter = mejor.user_attrs['best_iter']                 # iteraciones del mejor trial (sin validación no hay early stopping)

rows = []                                                 # un resultado por semilla
for seed in semillas[:n_semillas_test]:                   # las primeras 5 semillas de semillas.py
    params = {
        'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt', 'boost_from_average': True,
        'feature_pre_filter': False,
        **mejor.params,                                   # hiperparámetros del mejor trial de la optimización
        'bagging_freq': 1,                                # fijo, como en la optimización
        'seed': seed,                                     # lo único que cambia entre las 5 corridas
        'num_threads': lgb_threads,
        'verbose': -1
    }
    # Entrena con 201905-201908 (target fusionado) y las iteraciones del mejor trial; guarda el modelo de cada semilla
    model = lgb.train(params, data_test_train_lgb, num_boost_round=best_iter)
    model.save_model(modelos_path + f'EM_Baseline_Train0508_Test10_s{seed}.txt')
    y_pred = model.predict(X_test)                        # probabilidad de baja de cada cliente en octubre
    rows.append({
        'seed': seed,
        'ganancia': ganancia_real(y_pred, y_test),        # ganancia con umbral 0.025, solo BAJA+2 gana (y_test es la clase ternaria)
    })

df_test = pd.DataFrame(rows)                              # tabla con una fila por semilla

print(df_test)
print(f"Ganancia test 201910    : {df_test['ganancia'].mean():,.0f} ± {df_test['ganancia'].std():,.0f}")      # media ± desvío de las 5 semillas

        seed   ganancia
0  618541778  256272500
1  249611762  258142500
2  424089056  257400000
3  176108349  260287500
4  472407521  257400000
Ganancia test 201910    : 257,900,500 ± 1,492,290


##### Ganancia Acumulada en Octubre (5 semillas)

In [23]:
import plotly.graph_objects as go

limite = 20000  # cuántos clientes mostrar en el eje x

fig = go.Figure()
for seed in semillas[:n_semillas_test]:
  # Carga el modelo guardado de esta semilla y predice octubre
    model_seed = lgb.Booster(model_file=modelos_path + f'EM_Baseline_Train0508_Test10_s{seed}.txt')
    y_pred_seed = model_seed.predict(X_test)

  # Resultado real de cada cliente en ese orden: gana solo si fue BAJA+2
    orden = np.argsort(y_pred_seed)[::-1]                   # clientes de octubre de mayor a menor probabilidad  
    ganancia_ordenada = np.where(y_test.values[orden] == 'BAJA+2', ganancia_acierto, -costo_estimulo)
    ganancia_acumulada = np.cumsum(ganancia_ordenada)     # ganancia si se estimulan los primeros 1, 2, 3... clientes

    fig.add_trace(go.Scatter(
        x=list(range(1, limite + 1)), y=ganancia_acumulada[:limite],
        mode='lines', name=f'semilla {seed}'
    ))

fig.update_layout(
    title='Ganancia acumulada en Octubre según clientes estimulados - Baseline (5 semillas)',
    xaxis_title='Cantidad de clientes ordenados por probabilidad descendente',
    yaxis_title='Ganancia acumulada',
    xaxis_range=[0, limite],
    template='plotly_white'
)

fig.show()

##### Ganancia por Corte en Octubre (5 semillas)

In [ ]:
# Ganancia por corte fijo: 2.000, 2.500, ... 8.000 clientes, alrededor del máximo de la curva

cortes = np.arange(2000, 8001, 500)                       # cortes fijos (2.000, 2.500, ... 8.000) alrededor del máximo de la curva

rows_cortes = []                                          # ganancia de cada semilla en cada corte fijo
for seed in semillas[:n_semillas_test]:
  # Carga el modelo guardado de esta semilla y predice octubre
    model_seed = lgb.Booster(model_file=modelos_path + f'EM_Baseline_Train0508_Test10_s{seed}.txt')
    y_pred_seed = model_seed.predict(X_test)

  # Resultado real de cada cliente en ese orden: gana solo si fue BAJA+2
    orden = np.argsort(y_pred_seed)[::-1]                   # clientes de octubre de mayor a menor probabilidad
    ganancia_ordenada = np.where(y_test.values[orden] == 'BAJA+2', ganancia_acierto, -costo_estimulo)
    ganancia_acumulada = np.cumsum(ganancia_ordenada)     # ganancia si se estimulan los primeros 1, 2, 3... clientes

  # Ganancia de esta semilla en cada corte fijo (-1 porque la posición N está en el índice N-1)
    for corte in cortes:
        rows_cortes.append({
            "seed": seed,
            "corte": corte,
            "ganancia": ganancia_acumulada[corte - 1]
        })

df_ganancia_cortes = pd.DataFrame(rows_cortes)

# Estadísticas por corte sobre las 5 semillas
estadisticas_cortes = (
    df_ganancia_cortes.groupby('corte')['ganancia']
    .agg(['mean', 'std', 'min', 'max', 'median'])
)
estadisticas_cortes

,mean,std,min,max,median
corte,,,,,
2000,254540000.0,5.016772e+06,246400000,259600000,255200000.0
2500,270050000.0,3.648287e+06,264550000,274450000,271150000.0
3000,280060000.0,3.697702e+06,275000000,282700000,282700000.0
3500,285450000.0,3.300000e+06,279950000,288750000,286550000.0
4000,287540000.0,2.409979e+06,283800000,290400000,288200000.0
4500,288530000.0,3.681304e+06,284350000,293150000,288750000.0
5000,282700000.0,3.012474e+06,279400000,287100000,282700000.0
5500,281710000.0,2.972036e+06,276650000,284350000,282150000.0
6000,277860000.0,1.998249e+06,275000000,280500000,278300000.0


##### Boxplot de Ganancia por Corte en Octubre (5 Semillas)

In [25]:
# Boxplot de la ganancia por corte, con las 5 semillas y los percentiles 25 y 75 (q1 y q3) para las cajas

import plotly.graph_objects as go

# Estadísticas por corte sobre las 5 semillas, más q1 y q3 (percentiles 25 y 75) para las cajas
estadisticas_cortes = (
    df_ganancia_cortes.groupby('corte')['ganancia']
    .agg(['mean', 'std', 'min', 'max', 'median',
          lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)])
)
estadisticas_cortes.columns = ['mean', 'std', 'min', 'max', 'median', 'q1', 'q3']

fig = go.Figure()

# Un boxplot por corte, armado con las estadísticas ya calculadas
fig.add_trace(go.Box(
    x=[str(c) for c in estadisticas_cortes.index],         # corte como texto: cada uno es una caja separada
    q1=estadisticas_cortes['q1'],                          # borde inferior de la caja
    median=estadisticas_cortes['median'],                  # línea del medio
    q3=estadisticas_cortes['q3'],                          # borde superior de la caja
    lowerfence=estadisticas_cortes['min'],                 # bigote inferior: mínimo de las 5 semillas
    upperfence=estadisticas_cortes['max'],                 # bigote superior: máximo de las 5 semillas
    mean=estadisticas_cortes['mean'],                      # media (línea punteada)
    sd=estadisticas_cortes['std'],
    boxmean=True,
))

fig.update_layout(
    title=f'Distribución de ganancia por corte - Baseline ({n_semillas_test} semillas)',
    xaxis_title='Cantidad de clientes (corte)',
    yaxis_title='Ganancia acumulada',
    template='plotly_white'
)

fig.show()

##### Mejor Corte por Semilla y Mediana en Octubre

In [26]:
# Mejor corte de cada semilla: el N donde la ganancia acumulada llega a su máximo
rows_N = []
for seed in semillas[:n_semillas_test]:
  # Carga el modelo guardado de esta semilla y predice octubre
    model_seed = lgb.Booster(model_file=modelos_path + f'EM_Baseline_Train0508_Test10_s{seed}.txt')
    y_pred = model_seed.predict(X_test)

    orden = np.argsort(y_pred)[::-1]                   # clientes de octubre de mayor a menor probabilidad
    # Resultado real de cada cliente en ese orden: gana solo si fue BAJA+2
    ganancia_ordenada = np.where(y_test.values[orden] == 'BAJA+2', ganancia_acierto, -costo_estimulo)
    ganancia_acumulada = np.cumsum(ganancia_ordenada)     # ganancia si se estimulan los primeros 1, 2, 3... clientes

    mejor_N = int(np.argmax(ganancia_acumulada)) + 1      # +1 porque argmax empieza en 0
    rows_N.append({"seed": seed, "mejor_N": mejor_N, "mejor_ganancia": ganancia_acumulada[mejor_N - 1]})

df_mejorN = pd.DataFrame(rows_N)
print(df_mejorN)

# N central: mediana de los mejores N (uno por semilla), y su peso como % de los clientes de octubre
clientes_octubre = len(y_test.values)
N_central = df_mejorN['mejor_N'].median()
pct_central = N_central / clientes_octubre

# Barrido: el mismo porcentaje central, corrido -20%, -10%, 0, +10% y +20%
barrido = pd.DataFrame({'variacion': [-0.20, -0.10, 0.0, 0.10, 0.20]})
barrido['pct'] = (pct_central * (1 + barrido['variacion'])).round(4)
barrido['N_octubre'] = (barrido['pct'] * clientes_octubre).round().astype(int)     # clientes que implica cada porcentaje en octubre

print(f"Mediana del mejor N en octubre: {N_central:.0f}  ->  {pct_central*100:.2f}% de {clientes_octubre} clientes")
barrido

        seed  mejor_N  mejor_ganancia
0  618541778     4114       290565000
1  249611762     4112       292820000
2  424089056     4363       287017500
3  176108349     4159       295927500
4  472407521     4022       291995000
Mediana del mejor N en octubre: 4114  ->  3.03% de 135633 clientes


,variacion,pct,N_octubre
0,-0.2,0.0243,3296
1,-0.1,0.0273,3703
2,0.0,0.0303,4110
3,0.1,0.0334,4530
4,0.2,0.0364,4937


#### Resultados del Baseline

**Diseño:** optimización con train 201903–201906 (target BAJA+1 y BAJA+2 contra CONTINUA) y validación 201908 con el mismo target fusionado, AUC (50 trials). Reentrenamiento con 201905–201908 (target fusionado) y las iteraciones del mejor trial, 5 semillas. Test en 201910 con la ganancia contra BAJA+2 solamente.

**Optimización**
- Mejor trial: 32, con AUC de validación 0.9325 y 440 iteraciones.
- Hiperparámetros: `num_leaves` 119, `learning_rate` 0.014, `min_data_in_leaf` 920, `feature_fraction` 0.34, `bagging_fraction` 0.50.
- Los más influyentes en el AUC: `min_data_in_leaf` (0.39) y `learning_rate` (0.29). `feature_fraction` casi no pesa (0.04).
- El mejor AUC se estabilizó desde el trial 32 hasta el 49. El `learning_rate` quedó cerca del borde inferior del rango (0.01).

**Test en 201910 (media ± desvío de 5 semillas)**
- Ganancia con umbral 0.025: 257,9 M ± 1,5 M.

**Corte sugerido: 4.000 clientes (≈ 3% de los clientes de octubre)**
- Ganancia media de 287,5 M en 4.000, con el menor desvío de la zona del máximo (2,4 M). El máximo de la media está en 4.500 (288,5 M), pero la diferencia está dentro del ruido entre semillas.
- El mejor corte de cada semilla va de 4.022 a 4.363 clientes, con una mediana de 4.114 (3,03% de 135.633 clientes).

**Limitaciones**
- Con 5 semillas, el corte sugerido es poco preciso: el desvío de la ganancia por corte es de 2 a 4 M.
- El AUC de validación (target fusionado) y la ganancia de test (BAJA+2) miden cosas distintas, así que no se comparan entre sí.
- El train incluye meses con variables rotas (abril y mayo), y octubre comparte con mayo las variables rotas.